# AAT6153 Chapter 1 — Hands-on 5
## Classifying Three Animals with a CNN Trained from Scratch + Data Augmentation
**Slides:** Image Classification challenges (slide 34), CNN Layer Patterns (slides 51–52), Saving and Loading (slides 55–57), Data Augmentation (slide 58)

In this notebook you will:
1. Load a folder-based image dataset of three animal classes.
2. Use **data augmentation** layers to generate realistic variants of each training image.
3. Train a LeNet-style CNN **from scratch** on 150 × 150 colour images.
4. Evaluate with a confusion matrix, list misclassified images, then **save, reload and test** the model on your own photos.

*Runtime: Runtime ▸ Change runtime type ▸ **T4 GPU** required.*

## Download and extract the dataset

In [ ]:
!gdown 1hO_sRQwh4MxqBXEuwGe-2KVIQr7OaE8g -O three_animals_dataset.zip
!unzip -q -o three_animals_dataset.zip && echo "A total of $(find three_animals_dataset/ -type f | wc -l) files have been extracted."

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import image_dataset_from_directory, load_img, img_to_array
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

tf.keras.utils.set_random_seed(7)

train_dir = 'three_animals_dataset/train'
test_dir = 'three_animals_dataset/test'
IMG_SIZE = (150, 150)
BATCH_SIZE = 32

## Load the images
`image_dataset_from_directory` reads the folder structure `train/<class_name>/*.jpg`, so **the folder names become the class labels**.
10% of the training folder is held out for validation. Augmentation will only be applied to the training images.

In [ ]:
train_ds, val_ds = image_dataset_from_directory(
    train_dir, validation_split=0.1, subset='both', seed=7,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical',
    interpolation='bicubic')

test_ds = image_dataset_from_directory(
    test_dir, image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical',
    interpolation='bicubic', shuffle=False)

class_names = train_ds.class_names
test_files = test_ds.file_paths          # keep the file names (in order) for later
num_classes = len(class_names)
print("Classes:", class_names)

In [ ]:
# Look at some training images
images, labels = next(iter(train_ds))
plt.figure(figsize=(10, 10))
for i in range(9):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(images[i].numpy().astype('uint8'))
    plt.title(class_names[np.argmax(labels[i])]); plt.axis(False)
plt.show()

In [ ]:
# Cache and prefetch so the GPU is not left waiting for data
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)
test_ds = test_ds.cache().prefetch(AUTOTUNE)

---
## Data augmentation (slide 58)
Each augmentation layer applies a **random** transformation every time an image passes through it, so the network rarely sees exactly the same picture twice. This tackles the challenges on slide 34 (viewpoint, scale, deformation…) and reduces overfitting.
Augmentation layers are **only active during training**. They do nothing at prediction time.

In [ ]:
data_augmentation = models.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(30 / 360, fill_mode='nearest'),          # ±30 degrees
    layers.RandomTranslation(0.2, 0.2, fill_mode='nearest'),       # shift up to 20%
    layers.RandomZoom(0.2, fill_mode='nearest'),                   # zoom up to 20%
], name='data_augmentation')

# Show 9 augmented versions of the same training image
sample = images[0:1]
plt.figure(figsize=(10, 10))
for i in range(9):
    augmented = data_augmentation(sample, training=True)
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(augmented[0].numpy().astype('uint8')); plt.axis(False)
plt.suptitle("Nine random augmentations of one image"); plt.show()

---
## Construct the CNN
The same LeNet-style feature extractor as Hands-on 4, now with **3-channel 150 × 150** inputs and **3 output classes**.
Augmentation and pixel rescaling are built **into the model**, so the saved model accepts raw 0–255 images directly.

In [ ]:
model = models.Sequential([
    layers.Input(shape=IMG_SIZE + (3,)),
    data_augmentation,
    layers.Rescaling(1 / 255.),
    layers.Conv2D(6, (5, 5), padding='same', activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2), strides=(2, 2)),
    layers.Conv2D(16, (5, 5), padding='valid', activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2), strides=(2, 2)),
    layers.Flatten(),
    layers.Dense(120, activation='relu'),
    layers.Dense(84, activation='relu'),
    layers.Dense(num_classes, activation='softmax'),
])
model.summary()

**Discussion:** look at the parameter count of the first `Dense` layer. Why is it so large here compared with LeNet on MNIST? What could you add to the architecture to reduce it? (See the layer patterns on slide 52.)

## Compile and train

In [ ]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history = model.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=[early_stop])

In [ ]:
def plot_history(history):
    acc, val_acc = history.history['accuracy'], history.history['val_accuracy']
    loss, val_loss = history.history['loss'], history.history['val_loss']
    epochs = np.arange(1, len(acc) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(epochs, acc, 'b', label='train'); ax1.plot(epochs, val_acc, 'g', label='val')
    ax1.set(xlabel='Epoch', ylabel='Accuracy', title='Model Accuracy'); ax1.legend()
    ax2.plot(epochs, loss, 'b', label='train'); ax2.plot(epochs, val_loss, 'g', label='val')
    ax2.set(xlabel='Epoch', ylabel='Loss', title='Model Loss'); ax2.legend()
    plt.show()

plot_history(history)

## Evaluate on the test set

In [ ]:
score = model.evaluate(test_ds, verbose=0)
print(f'Loss: {score[0]:.4f}, accuracy: {score[1]:.2%}')

y_true = np.concatenate([np.argmax(y, axis=1) for _, y in test_ds])
y_pred = np.argmax(model.predict(test_ds, verbose=0), axis=1)

print(classification_report(y_true, y_pred, target_names=class_names, digits=4, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_true, y_pred, display_labels=class_names, cmap='Blues')
plt.title('Confusion matrix (CNN from scratch)'); plt.show()

In [ ]:
# List files that were misclassified
errors = [(test_files[i], class_names[y_true[i]], class_names[y_pred[i]])
          for i in np.where(y_true != y_pred)[0]]
print(f"{len(errors)} misclassified test images")
for path, true, pred in errors[:20]:
    print(f"  {path}  (true: {true}, predicted: {pred})")

---
## Save and reload the model (slides 55–57)

In [ ]:
model.save('three_animals_scratch.keras')
with open('class_names.json', 'w') as f:
    json.dump(class_names, f)
print("Saved model to disk")

In [ ]:
model = tf.keras.models.load_model('three_animals_scratch.keras')
with open('class_names.json') as f:
    class_names = json.load(f)
print("Model loaded from disk. Classes:", class_names)

## Test the model on your own images
Upload one or more photos. An image is labelled **None** if no class has a probability above 0.5.

In [ ]:
from google.colab import files
uploaded = files.upload()

for fn in uploaded:
    img = load_img(fn, target_size=IMG_SIZE)
    x = np.expand_dims(img_to_array(img), axis=0)      # raw 0–255; Rescaling is inside the model
    probs = model.predict(x, verbose=0)[0]

    for name, p in zip(class_names, probs):
        print(f'{name:10}: {p: .3f}')
    predicted_class = class_names[probs.argmax()] if probs.max() > 0.5 else "None"

    plt.axis(False)
    plt.imshow(load_img(fn))
    plt.title(f'Predicted as {predicted_class}')
    plt.show()

---
## Try it yourself
1. **No augmentation:** remove `data_augmentation` from the model and retrain. Compare the gap between training and validation accuracy.
2. **Deeper network (slide 52):** use the pattern `INPUT → [CONV → ACT → POOL] × 4 → FC → ACT → DO → FC` with 32, 64, 128, 128 filters (3 × 3). How do the parameter count and accuracy change?
3. **Global average pooling (slide 49):** replace `Flatten()` with `GlobalAveragePooling2D()`. What happens to the parameter count?
4. Keep your best model's test accuracy. You will compare it with **transfer learning** in Hands-on 6.

In [ ]:
# Use this cell for your experiments